In [1]:
import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import re
import time

In [2]:
def set_up_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [3]:
def get_product_information(driver, label):
    try:
        # Tìm tất cả p tags có class text-sm (chứa label)
        labels = driver.find_elements(By.CSS_SELECTOR, "p[class*='text-sm'][class*='leading-14']")
        
        for label_el in labels:
            label_text = label_el.text.strip()
            
            if label.lower() in label_text.lower():
                # Lấy parent div grid rồi tìm value trong sibling
                parent = label_el.find_element(By.XPATH, "./ancestor::div[contains(@class, 'grid')][1]")
                # Tìm div chứa text value (không phải label)
                value_els = parent.find_elements(By.CSS_SELECTOR, "div")
                
                for val in value_els:
                    val_text = val.text.strip()
                    # Lấy value khác label
                    if val_text and val_text != label_text:
                        return val_text
    except:
        return None

In [4]:
def get_section_detail(driver, section_id):
    try:
        # Tìm element theo id
        heading = driver.find_element(By.ID, section_id)
        
        # Lấy siblings trực tiếp từ heading
        siblings = heading.find_elements(By.XPATH, "./following-sibling::*")
        
        content = []
        for sib in siblings:
            tag = sib.tag_name.lower()
            
            # Dừng khi gặp heading tiếp theo (h2 hoặc h3)
            if tag in ["h2", "h3"]:
                break
            
            # Lấy text
            t = sib.get_attribute("textContent") or ""
            t = t.strip()
            if t:
                content.append(t)
        
        return "\n".join(content) if content else None
    
    except:
        return None

In [5]:
def get_product_images(driver):
    images = []
    
    try:
        # Tìm tất cả img elements trong product detail
        img_elements = driver.find_elements(By.CSS_SELECTOR, "img[src*='production-cdn'], img[src*='pharmacity']")
        
        for img in img_elements:
            src = img.get_attribute("src")
            
            # Nếu src rỗng, thử lấy từ srcset hoặc data-src
            if not src:
                src = img.get_attribute("srcset") or img.get_attribute("data-src")
            
            if src:
                # Extract URL đầy đủ từ production-cdn.pharmacity.io
                match = re.search(r'(https://production-cdn\.pharmacity\.io/[^\s]+)', src)
                if match:
                    url = match.group(1).split()[0]  # Loại bỏ size info nếu có
                    if url not in images:
                        images.append(url)
                # Hoặc lấy trực tiếp nếu là URL hợp lệ
                elif src.startswith('http') and 'pharmacity' in src:
                    if src not in images:
                        images.append(src)
    
    except Exception as e:
        print(f"Error getting images: {e}")
    
    return images

In [6]:
def get_name(driver):
    selectors = [
        "h2[class*='text-32'][class*='semibold']",  # CSS selector
        "#pmc-v2 h2.text-\\[32px\\]",  # CSS với class cụ thể
        "//*[@id='pmc-v2']//h2[contains(@class, 'font-semibold')]"  # XPATH fallback
    ]
    time.sleep(3)  
    for selector in selectors:
        try:
            if selector.startswith("//"):  # XPATH
                name = driver.find_element(By.XPATH, selector).text.strip()
            else:  # CSS
                name = driver.find_element(By.CSS_SELECTOR, selector).text.strip()
            
            if name and name.lower() != 'none':
                return name
        except:
            continue
    
    return None

In [7]:
# Sửa lại XPATH nút 'Xem thêm'
def click_view_more(driver):
    
    try:
        time.sleep(1)
        btn_section = driver.find_element(By.XPATH,"//button[@aria-label='Xem thêm']")
        time.sleep(1)
        btn = btn_section.find_element(By.TAG_NAME, "button")
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
        time.sleep(1)
        btn.click()
        time.sleep(1)
    except Exception as e:
        print(f"Error clicking 'Xem thêm' button: {e}")
    

In [8]:
def get_medicine_data(driver):
    try:
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-cols-1.gap-6.lg\:grid-cols-2.lg\:items-start > div.grid.grid-cols-1.gap-4 > div > div > div.flex.flex-col.gap-4")))
    except:
        pass

    try:
        name = get_name(driver)
        time.sleep(1)
    except:
        name = None
    
    try:
        price = driver.find_element(By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-cols-1.gap-6.lg\:grid-cols-2.lg\:items-start > div.grid.grid-cols-1.gap-4 > div > div > div.flex.flex-col.gap-4 > div.rounded-xs.bg-surface-canvas > div > div.flex.flex-row.items-center.gap-2 > h3").text.strip()
        time.sleep(1)
    except:
        price = None

    images = get_product_images(driver)
    time.sleep(1)

    nha_san_xuat = get_product_information(driver, "Nhà sản xuất:")
    noi_san_xuat = get_product_information(driver, "Nơi sản xuất:")
    thuoc_can_ke_don = get_product_information(driver, "Thuốc cần kê đơn:")
    hoat_chat = get_product_information(driver,"Hoạt chất:")
    chi_dinh_tom_tat = get_product_information(driver, "Chỉ định:")
    dang_bao_che = get_product_information(driver, "Dạng bào chế:")
    quy_cach = get_product_information(driver, "Quy cách:")
    luu_y = get_product_information(driver, "Lưu ý:")
    


    mo_ta = get_section_detail(driver, "mo-ta")
    thanh_phan = get_section_detail(driver, "thanh-phan")
    chi_dinh_chi_tiet = get_section_detail(driver, "chi-dinh")
    huong_dan_su_dung = get_section_detail(driver, "huong-dan-su-dung")
    cach_su_dung = get_section_detail(driver, "cach-su-dung")
    than_trong = get_section_detail(driver, "than-trong")
    thong_tin_san_xuat = get_section_detail(driver, "thong-tin-san-xuat")

    return {
        "name": name,
        "price": price,
        "images": images,
        "nha_san_xuat": nha_san_xuat,
        "noi_san_xuat": noi_san_xuat,
        "thuoc_can_ke_don": thuoc_can_ke_don,
        "hoat_chat": hoat_chat,
        "chi_dinh_tom_tat": chi_dinh_tom_tat,
        "dang_bao_che": dang_bao_che,
        "quy_cach": quy_cach,
        "luu_y": luu_y,
        "mo_ta": mo_ta,
        "thanh_phan": thanh_phan,
        "chi_dinh_chi_tiet": chi_dinh_chi_tiet,
        "huong_dan_su_dung": huong_dan_su_dung,
        "cach_su_dung": cach_su_dung,
        "than_trong": than_trong,
        "thong_tin_san_xuat": thong_tin_san_xuat
    }

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:14: SyntaxWarning: invalid escape sequence '\['
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:14: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_4044\96172029.py:3: SyntaxWarning: invalid escape sequence '\['
  WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-cols-1.gap-6.lg\:grid-cols-2.lg\:items-start > div.grid.grid-cols-1.gap-4 > div > div > div.flex.flex-col.gap-4")))
C:\Users\Admin\AppData\Local\Temp\ipykernel_4044\96172029.py:14: SyntaxWarning: invalid escape sequence '\['
  price = driver.find_element(By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-cols-1.gap-6.lg\:grid-cols-2.lg\:items-start > div.grid.grid-cols-1.gap-4 > div > div > div.flex.flex-col.ga

In [9]:
error_df = pd.read_csv("pharmacity_medicines_still_error.csv")
error_df.shape
error_df.head()

,name,price,images,nha_san_xuat,noi_san_xuat,thuoc_can_ke_don,hoat_chat,chi_dinh_tom_tat,dang_bao_che,quy_cach,...,huong_dan_su_dung,cach_su_dung,than_trong,thong_tin_san_xuat,url,category,drug_type,category_link,category_name,error
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/thuoc-cam-lanh#,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/nasol-spray-chai-70m...,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/tinh-dau-tre-em-naso...,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/nhom-san-pham-giam-d...,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/thuoc-da-lieu#,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...


In [10]:
# Thực hiện crawl lại các link bị lỗi
if len(error_df) > 0:
    driver = None
    results = []
    retry_failed = []  # Lưu link vẫn lỗi để retry lần sau
    
    try:
        driver = set_up_driver()
        
        for i, row in error_df.iterrows():
            url = row['url']  
            drug_type = row.get('drug_type') if hasattr(row, 'get') else row.get('drug_type', '')
            category_link = row.get('category_link') if hasattr(row, 'get') else row.get('category_link', '')
            
            try:
                print(f"[{i+1}/{len(error_df)}] Crawling: {url}")
                driver.get(url)
                time.sleep(3)
                
                data = get_medicine_data(driver)
                
                # Kiểm tra xem data có hợp lệ không (name không rỗng)
                if data.get('name') and str(data.get('name')).lower() != 'none':
                    data['url'] = url
                    data['drug_type'] = drug_type
                    data['category_link'] = category_link
                    results.append(data)
                    print(f"Crawled successfully: {data.get('name')}")
                else:
                    print(f"Invalid data (name empty): {url}")
                    retry_failed.append(url)
                    
            except Exception as e:
                print(f"Error crawling {url}: {e}")
                retry_failed.append(url)
                
                # Nếu lỗi liên tục, khởi tạo lại driver
                if len(retry_failed) > 3:
                    print("Reinitializing driver...")
                    driver.quit()
                    driver = set_up_driver()
                    retry_failed = []
    
    finally:
        if driver is not None:
            driver.quit()
    
    # Lưu kết quả
    if len(results) > 0:
        df_recrawled = pd.DataFrame(results)
        df_recrawled.to_csv(
            "pharmacity_medicines_error_fixed.csv",
            index=False,
            encoding='utf-8'
        )
        print(f"\Saved {len(results)} successfully crawled records")
    
    # Lưu các link vẫn lỗi để crawl lại lần sau
    if len(retry_failed) > 0:
        error_df_retry = error_df[error_df['url'].isin(retry_failed)]
        error_df_retry.to_csv(
            "pharmacity_medicines_still_error(2).csv",
            index=False,
            encoding='utf-8'
        )
        print(f"{len(retry_failed)} links still have errors - saved to 'still_error(2).csv'")
    
    print(f"\nSummary:")
    print(f"- Successfully recrawled: {len(results)}")
    print(f"- Still failed: {len(retry_failed)}")

<>:56: SyntaxWarning: invalid escape sequence '\S'
<>:56: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Admin\AppData\Local\Temp\ipykernel_4044\2505010732.py:56: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Saved {len(results)} successfully crawled records")


[1/39] Crawling: https://www.pharmacity.vn/thuoc-cam-lanh#
Invalid data (name empty): https://www.pharmacity.vn/thuoc-cam-lanh#
[2/39] Crawling: https://www.pharmacity.vn/nasol-spray-chai-70ml.html
Crawled successfully: Thuốc xịt mũi Nasol Spray giảm triệu chứng nghẹt mũi, sổ mũi, viêm mũi dị ứng (chai 70ml)
[3/39] Crawling: https://www.pharmacity.vn/tinh-dau-tre-em-nasomom-4-spray-chai-70ml.html
Crawled successfully: Thuốc xịt mũi Nasomom 4 Tinh dầu trẻ em giảm sổ mũi, viêm mũi dị ứng, khò khè trẻ nhỏ (chai 70ml)
[4/39] Crawling: https://www.pharmacity.vn/nhom-san-pham-giam-dau-da-day-cho-nguoi-lon.html
Crawled successfully: Nhóm sản phẩm giảm đau dạ dày cho người lớn
[5/39] Crawling: https://www.pharmacity.vn/thuoc-da-lieu#
Invalid data (name empty): https://www.pharmacity.vn/thuoc-da-lieu#
[6/39] Crawling: https://www.pharmacity.vn/bacterocin-5g.html
Crawled successfully: Thuốc dùng ngoài Bacterocin điều trị tại chỗ bệnh chốc lỡ, viêm nang lông và mụn mủ (tuýp 5g)
[7/39] Crawling: h

In [ ]:

error_df = pd.read_csv("pharmacity_medicines_still_error(2).csv")
print(error_df.shape)
error_df.head()

(16, 24)


,name,price,images,nha_san_xuat,noi_san_xuat,thuoc_can_ke_don,hoat_chat,chi_dinh_tom_tat,dang_bao_che,quy_cach,...,huong_dan_su_dung,cach_su_dung,than_trong,thong_tin_san_xuat,url,category,drug_type,category_link,category_name,error
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/thuoc-cam-lanh#,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/thuoc-da-lieu#,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/thuoc-than-kinh#,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/thuoc-co-xuong-khop#,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/vitamin-va-khoang-chat#,NaN,NaN,NaN,NaN,Message: no such element: Unable to locate ele...


In [15]:
fix_df = pd.read_csv("pharmacity_medicines_error_fixed.csv")
print(fix_df.shape)
fix_df.head()

(23, 21)


,name,price,images,nha_san_xuat,noi_san_xuat,thuoc_can_ke_don,hoat_chat,chi_dinh_tom_tat,dang_bao_che,quy_cach,...,mo_ta,thanh_phan,chi_dinh_chi_tiet,huong_dan_su_dung,cach_su_dung,than_trong,thong_tin_san_xuat,url,drug_type,category_link
0,Thuốc xịt mũi Nasol Spray giảm triệu chứng ngh...,30.000 ₫/Chai,['https://prod-cdn.pharmacity.io/e-com/images/...,RELIV HEALTHCARE,NaN,NaN,Natri Clorid 630mg,"Giảm triệu chứng nghẹt mũi, sổ mũi, Viêm mũi d...",Thuốc xịt mũi,Chai 70ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/nasol-spray-chai-70m...,NaN,NaN
1,Thuốc xịt mũi Nasomom 4 Tinh dầu trẻ em giảm s...,39.000 ₫/Chai,['https://prod-cdn.pharmacity.io/e-com/images/...,RELIV HEALTHCARE,NaN,NaN,Natri Clorid 630mg,"Giảm triệu chứng nghẹt mũi, sổ mũi, Viêm mũi d...",Tinh dầu xịt mũi,Chai 70ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/tinh-dau-tre-em-naso...,NaN,NaN
2,Nhóm sản phẩm giảm đau dạ dày cho người lớn,220.900 ₫/Bộ,['https://prod-cdn.pharmacity.io/e-com/images/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/nhom-san-pham-giam-d...,NaN,NaN
3,Thuốc dùng ngoài Bacterocin điều trị tại chỗ b...,47.000 ₫/Tuýp,['https://prod-cdn.pharmacity.io/e-com/images/...,Genuone Sciences Inc.,NaN,Không,Mupirocin,"Bệnh chốc lở, viêm nang lông và mụn mủ do Stap...",Thuốc mỡ,NaN,...,NaN,Công Dụng\nChỉ định\nThuốc Bacterocin Oint chỉ...,NaN,Tác dụng phụ\n\nCác tác dụng không mong muốn k...,NaN,NaN,NaN,https://www.pharmacity.vn/bacterocin-5g.html,NaN,NaN
4,Thuốc dùng ngoài Bacterocin điều trị tại chỗ b...,105.000 ₫/Tuýp,['https://prod-cdn.pharmacity.io/e-com/images/...,Genuone Sciences Inc.,Viet Nam,Không,Mupirocin,"Chốc lở, viêm nang lông và mụn mủ do S.aureus ...",Thuốc mỡ,Tuýp 15g,...,NaN,Công dụng\nChỉ định\nThuốc Bacterocin Oint chỉ...,NaN,Tác dụng phụ\nCác tác dụng không mong muốn khi...,NaN,NaN,NaN,https://www.pharmacity.vn/bacterocin-15g.html,NaN,NaN


In [17]:
recrawled_df = pd.read_csv("pharmacity_medicines_recrawled.csv")
print(recrawled_df.shape)
recrawled_df.head()

(454, 21)


,name,price,images,nha_san_xuat,noi_san_xuat,thuoc_can_ke_don,hoat_chat,chi_dinh_tom_tat,dang_bao_che,quy_cach,...,mo_ta,thanh_phan,chi_dinh_chi_tiet,huong_dan_su_dung,cach_su_dung,than_trong,thong_tin_san_xuat,url,drug_type,category_link
0,Viên nén Desloratadine 5mg trị viêm mũi dị ứng...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,Genepharm.SA,Greece,Không,Desloratadin 5mg,"Điều trị Viêm mũi dị ứng, mày đay, ngứa",Viên nén bao phim,2 vỉ x 15 viên,...,Thành phần của Thuốc Desloratadine 5mg\n\nHoạt...,Công dụng của Thuốc Desloratadine 5mg\nChỉ địn...,Cách sử dụng\nCách dùng\n\nViên nén dùng theo ...,Tác dụng phụ\nDesloratadine thường được dung n...,NaN,"Bảo quản\nBảo quản thuốc trong bao bì kín, nơi...",NaN,https://www.pharmacity.vn/desloratadine-5mg-ho...,NaN,NaN
1,Azicine Stella 500mg điều trị các trường hợp n...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,STELLA,NaN,NaN,Azithromycin (dưới dạng azithromycin dihydrate),Điều trị các trường hợp nhiễm khuẩn gây ra bởi...,Viên nén bao phim,Hộp 6 viên,...,NaN,Chỉ định (Thuốc dùng cho bệnh gì?)\nĐiều trị c...,Hướng dẫn sử dụng\nLiều dùng\nNgười lớn và trẻ...,Chống chỉ định (Khi nào không nên dùng thuốc n...,NaN,"Thông tin sản xuất\nBảo quản: Nơi khô, nhiệt đ...",NaN,https://www.pharmacity.vn/azicine-500-stella-1...,NaN,NaN
2,Viên nang Kodemin trị ho khan do kích thích (1...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,Duoc VTYT Hai Duong,NaN,NaN,Codein phosphat; Guaifenesin,Điều trị ho khan do kích thích,Viên nang mềm,10 vỉ x 10 viên,...,Thành phần Viên nang Kodemin\n\nCodein phospha...,Công dụng của Viên nang Kodemin\nChỉ định (Thu...,Cách sử dụng Viên nang Kodemin\nLiều dùng và c...,Tác dụng phụ của Viên nang Kodemin\nThường gặp...,NaN,Thông tin sản xuất của Viên nang Kodemin\nThươ...,NaN,https://www.pharmacity.vn/kodemin-softcap-10vi...,NaN,NaN
3,"Viên nang A.T Arginin 200mg hạ men gan, tăng c...",NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,An Thien,NaN,NaN,Arginin hydroclorid 200mg,Hỗ trợ điều trị các rối loạn khó tiêu,Viên nang,Chai 85ml,...,NaN,Chỉ định (Thuốc dùng cho bệnh gì?)\n- Điều trị...,Liều dùng \n- Điều trị duy trì tăng amoniac má...,Chống chỉ định (Khi nào không nên dùng thuốc n...,NaN,"Thông tin sản xuất\nBảo quản: Nơi khô, nhiệt đ...",NaN,https://www.pharmacity.vn/at-arginin-200mg-10-...,NaN,NaN
4,Bột pha tiêm Neutrivit 5000 Bidiphar viêm dây ...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,BIDIPHAR,NaN,Không,Vitamin B1;Vitamin B6;Vitamin B12,"Viêm dây thần kinh, đau thần kinh tọa, thiếu v...",Bột đông khô pha tiêm,4 lọ bột + 4 ống dung môi,...,NaN,Công dụng của Bột pha tiêm Neutrivit 5000\n\n...,Cách dùng Bột pha tiêm Neutrivit 5000\n\nCách ...,Tác dụng phụ\n\nKhi sử dụng thuốc Neutrivit 50...,NaN,"Bảo quản\n\nNơi mát, nhiệt độ < 30°C, kín, trá...",NaN,https://www.pharmacity.vn/neutrivit-5000-bidip...,NaN,NaN


In [18]:
# Gộp dữ liệu đã fix với dữ liệu cũ (nếu cần) và lưu lại file tổng hợp
final_df = pd.concat([fix_df, recrawled_df], ignore_index=True)
final_df.to_csv("pharmacity_medicines_recrawled_fixed.csv", index=False, encoding='utf-8')

In [ ]:
final_df = pd.read_csv("pharmacity_medicines_recrawled_fixed.csv")
print(final_df.shape)
final_df.head()

(477, 21)


,name,price,images,nha_san_xuat,noi_san_xuat,thuoc_can_ke_don,hoat_chat,chi_dinh_tom_tat,dang_bao_che,quy_cach,...,mo_ta,thanh_phan,chi_dinh_chi_tiet,huong_dan_su_dung,cach_su_dung,than_trong,thong_tin_san_xuat,url,drug_type,category_link
0,Thuốc xịt mũi Nasol Spray giảm triệu chứng ngh...,30.000 ₫/Chai,['https://prod-cdn.pharmacity.io/e-com/images/...,RELIV HEALTHCARE,NaN,NaN,Natri Clorid 630mg,"Giảm triệu chứng nghẹt mũi, sổ mũi, Viêm mũi d...",Thuốc xịt mũi,Chai 70ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/nasol-spray-chai-70m...,NaN,NaN
1,Thuốc xịt mũi Nasomom 4 Tinh dầu trẻ em giảm s...,39.000 ₫/Chai,['https://prod-cdn.pharmacity.io/e-com/images/...,RELIV HEALTHCARE,NaN,NaN,Natri Clorid 630mg,"Giảm triệu chứng nghẹt mũi, sổ mũi, Viêm mũi d...",Tinh dầu xịt mũi,Chai 70ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/tinh-dau-tre-em-naso...,NaN,NaN
2,Nhóm sản phẩm giảm đau dạ dày cho người lớn,220.900 ₫/Bộ,['https://prod-cdn.pharmacity.io/e-com/images/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/nhom-san-pham-giam-d...,NaN,NaN
3,Thuốc dùng ngoài Bacterocin điều trị tại chỗ b...,47.000 ₫/Tuýp,['https://prod-cdn.pharmacity.io/e-com/images/...,Genuone Sciences Inc.,NaN,Không,Mupirocin,"Bệnh chốc lở, viêm nang lông và mụn mủ do Stap...",Thuốc mỡ,NaN,...,NaN,Công Dụng\nChỉ định\nThuốc Bacterocin Oint chỉ...,NaN,Tác dụng phụ\n\nCác tác dụng không mong muốn k...,NaN,NaN,NaN,https://www.pharmacity.vn/bacterocin-5g.html,NaN,NaN
4,Thuốc dùng ngoài Bacterocin điều trị tại chỗ b...,105.000 ₫/Tuýp,['https://prod-cdn.pharmacity.io/e-com/images/...,Genuone Sciences Inc.,Viet Nam,Không,Mupirocin,"Chốc lở, viêm nang lông và mụn mủ do S.aureus ...",Thuốc mỡ,Tuýp 15g,...,NaN,Công dụng\nChỉ định\nThuốc Bacterocin Oint chỉ...,NaN,Tác dụng phụ\nCác tác dụng không mong muốn khi...,NaN,NaN,NaN,https://www.pharmacity.vn/bacterocin-15g.html,NaN,NaN
